In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

prices_path = Path("Data/train_prices_15min.csv")
prices = pd.read_csv(prices_path, parse_dates=["delivery_date"])

display(prices.head())


,delivery_date,market_id,hour_index,quarter_index,price_eur_mwh
0,2025-10-01,ZONE_0,1,1,102.60
1,2025-10-01,ZONE_0,1,2,92.24
2,2025-10-01,ZONE_0,1,3,86.03
3,2025-10-01,ZONE_0,1,4,85.39
4,2025-10-01,ZONE_0,2,1,89.99


## 1. Dataset overview

In [2]:
dataset_overview = pd.Series({
    "rows": len(prices),
    "zones": prices["market_id"].nunique(),
    "first_delivery_date": prices["delivery_date"].min(),
    "last_delivery_date": prices["delivery_date"].max(),
    "missing_prices": prices["price_eur_mwh"].isna().sum(),
}, name="value")

display(dataset_overview.to_frame())

zone_summary = (
    prices.groupby("market_id", dropna=False)
    .agg(
        observations=("price_eur_mwh", "size"),
        price_count=("price_eur_mwh", "count"),
        avg_price_eur_mwh=("price_eur_mwh", "mean"),
        median_price_eur_mwh=("price_eur_mwh", "median"),
        min_price_eur_mwh=("price_eur_mwh", "min"),
        max_price_eur_mwh=("price_eur_mwh", "max"),
    )
    .sort_index()
)
zone_summary.index.name = "zone"

display(zone_summary.round(2))


,value
rows,139392
zones,4
first_delivery_date,2025-10-01 00:00:00
last_delivery_date,2026-09-30 00:00:00
missing_prices,0


,observations,price_count,avg_price_eur_mwh,median_price_eur_mwh,min_price_eur_mwh,max_price_eur_mwh
zone,,,,,,
ZONE_0,34848,34848,104.42,104.13,-499.99,747.10
ZONE_1,34848,34848,119.81,120.32,-498.65,645.20
ZONE_2,34848,34848,138.31,134.50,0.00,349.99
ZONE_3,34848,34848,76.75,78.00,-498.65,487.38


## 2. Feature engineering

One row represents one quarter-hour in one zone. Join keys are `delivery_date`, `market_id`, `hour_index` and `quarter_index`. Predictor groups are defined once in `FEATURE_GROUPS`; the same groups are used in the experiment table and both models.

| Group | Columns | Meaning |
| --- | --- | --- |
| `position` | `position_zone`, `position_hour`, `position_quarter`, `position_quarter_of_day` | Zone category; hour 1–24; quarter within hour 1–4; quarter within day 1–96. Hour 1 begins at midnight. |
| `calendar` | `calendar_day_of_week`, `calendar_is_weekend`, `calendar_month`, `calendar_day_of_year` | Weekday (Monday = 0), weekend flag, month 1–12 and day of year. |
| `calendar` | `calendar_month_sin`, `calendar_month_cos` | Sine/cosine of `2π × (month − 1) / 12`, keeping December and January close in the annual cycle. |
| `calendar` | `calendar_year_sin`, `calendar_year_cos` | Sine/cosine of the fraction of the year elapsed, using 365/366 days. |
| `local` | `local_hourly_price_eur_mwh`, `local_prev_hour_price_eur_mwh`, `local_next_hour_price_eur_mwh` | Current, previous and next known hourly prices in the same day/zone. At day boundaries, the unavailable neighbor uses the current price. |
| `daily` | `daily_{mean,min,max,range,std}_price_eur_mwh` | Statistics of the 24 known hourly prices. Range = max − min; standard deviation uses `ddof=0`. |
| `daily` | `daily_hour_01_price_eur_mwh` … `daily_hour_24_price_eur_mwh` | Full known hourly curve, repeated on each row for that day/zone. |
| `reference` | `reference_smooth_price_eur_mwh` | Smooth quarter-hour reference from mean-preserving spline interpolation of hourly averages, implemented with SciPy. |
| `history_hourly` | `history_hourly_deviation_lag_{1,7}d_eur_mwh` | Price minus hourly average at the same zone/hour/quarter exactly 1 or 7 calendar days earlier. |
| `history_smooth` | `history_smooth_deviation_lag_{1,7}d_eur_mwh` | Price minus smooth reference at the same zone/hour/quarter exactly 1 or 7 calendar days earlier. |
| `history_hourly`, `history_smooth` | `history_{hourly,smooth}_deviation_{mean,median}_{7,28}d_eur_mwh` | Mean/median residual over the preceding 7/28 calendar days, excluding today. |
| `history_hourly`, `history_smooth` | `history_{hourly,smooth}_deviation_count_{7,28}d` | Number of observations contributing to each historical window. |
| `crosszone` | `crosszone_ZONE_{0,1,2,3}_hourly_price_eur_mwh` | Each zone's known hourly price at the same date/hour. |
| `crosszone` | `crosszone_ZONE_{0,1,2,3}_minus_local_eur_mwh` | Each zone's hourly price minus the current row's local hourly price. |

**References and labels.** `reference_hourly_price_eur_mwh` is retained to calculate hourly residuals and reconstruct prices; it equals `local_hourly_price_eur_mwh`, so it is excluded from predictors. The labels are `price_eur_mwh`, `target_hourly_deviation_eur_mwh = price − hourly reference`, and `target_smooth_deviation_eur_mwh = price − smooth reference`. Labels never enter model inputs. Predict one residual type at a time and add it to its matching reference, followed by the hourly correction.

**History and missing values.** Use strictly earlier dates, with calendar-date joins for lags. Missing historical values remain `NaN` and counts are zero with no history. Ridge uses training-only median imputation and missingness indicators; CatBoost handles numeric missingness natively.

In [3]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline

DAY_KEYS = ["delivery_date", "market_id"]
HOUR_KEYS = DAY_KEYS + ["hour_index"]
QUARTER_KEYS = HOUR_KEYS + ["quarter_index"]
FEATURE_PREFIXES = (
    "position_", "calendar_", "local_", "daily_",
    "reference_", "history_", "crosszone_",
)


REFERENCE_COLUMNS = {
    "hourly": "reference_hourly_price_eur_mwh",
    "smooth": "reference_smooth_price_eur_mwh",
}
TARGET_COLUMNS = {
    "hourly": "target_hourly_deviation_eur_mwh",
    "smooth": "target_smooth_deviation_eur_mwh",
}


def hourly_from_quarters(quarter_prices):
    """Recreate the challenge's hourly inputs from complete observed hours."""
    data = quarter_prices.copy()
    data["delivery_date"] = pd.to_datetime(data["delivery_date"])
    if data.duplicated(QUARTER_KEYS).any():
        raise ValueError("Duplicate quarter-hour keys.")
    if not data["quarter_index"].isin([1, 2, 3, 4]).all():
        raise ValueError("quarter_index must be 1, 2, 3 or 4.")
    if not np.isfinite(data["price_eur_mwh"]).all():
        raise ValueError("Observed prices must be finite.")
    if not data.groupby(HOUR_KEYS).size().eq(4).all():
        raise ValueError("Each observed hour must contain all four quarters.")
    return (
        data.groupby(HOUR_KEYS, as_index=False)["price_eur_mwh"]
        .mean()
        .rename(columns={"price_eur_mwh": "hourly_price_eur_mwh"})
    )


def smooth_reference(hourly_prices):
    """Expand 24 hourly averages into 96 smooth, hourly-consistent values."""
    hourly = hourly_prices.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    if hourly.duplicated(HOUR_KEYS).any():
        raise ValueError("Duplicate hourly keys.")
    curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    curve = curve.reindex(columns=range(1, 25)).sort_index()
    if not np.isfinite(curve.to_numpy()).all():
        raise ValueError("Every day/zone needs 24 finite hourly prices.")
    if not hourly["hour_index"].isin(range(1, 25)).all():
        raise ValueError("hour_index must be between 1 and 24.")

    # Integral over a one-hour interval equals its hourly average × 1 hour.
    cumulative = np.column_stack([
        np.zeros(len(curve)), np.cumsum(curve.to_numpy(), axis=1)
    ])
    integral_spline = CubicSpline(
        np.arange(25), cumulative, axis=1, bc_type="natural", extrapolate=False
    )
    quarter_boundaries = np.arange(97) / 4
    reference = np.diff(integral_spline(quarter_boundaries), axis=1) * 4

    result = curve.index.to_frame(index=False).loc[
        curve.index.to_frame(index=False).index.repeat(96)
    ].reset_index(drop=True)
    result["hour_index"] = np.tile(np.repeat(np.arange(1, 25), 4), len(curve))
    result["quarter_index"] = np.tile(np.arange(1, 5), len(curve) * 24)
    result["reference_smooth_price_eur_mwh"] = reference.reshape(-1)
    np.testing.assert_allclose(
        reference.reshape(-1, 24, 4).mean(axis=2), curve.to_numpy(), atol=1e-8, rtol=0
    )
    return result

def build_features(hourly_prices, history_15min, target_prices=None):
    """Build features from known hourly inputs and strictly earlier history.

    target_prices is optional and adds labels only; it never feeds into X.
    Historical inputs may contain many dates: every historical feature is
    filtered by the date of its individual prediction row.
    """
    hourly = hourly_prices.copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    hourly = hourly.sort_values(HOUR_KEYS).reset_index(drop=True)
    features = smooth_reference(hourly)

    # Position and calendar.
    features["position_zone"] = features["market_id"].astype("category")
    features["position_hour"] = features["hour_index"]
    features["position_quarter"] = features["quarter_index"]
    features["position_quarter_of_day"] = (
        (features["hour_index"] - 1) * 4 + features["quarter_index"]
    )
    dates = features["delivery_date"].dt
    features["calendar_day_of_week"] = dates.dayofweek
    features["calendar_is_weekend"] = (dates.dayofweek >= 5).astype(int)
    features["calendar_month"] = dates.month
    month_angle = 2 * np.pi * (dates.month - 1) / 12
    features["calendar_month_sin"] = np.sin(month_angle)
    features["calendar_month_cos"] = np.cos(month_angle)
    features["calendar_day_of_year"] = dates.dayofyear
    year_fraction = (dates.dayofyear - 1) / np.where(dates.is_leap_year, 366, 365)
    features["calendar_year_sin"] = np.sin(2 * np.pi * year_fraction)
    features["calendar_year_cos"] = np.cos(2 * np.pi * year_fraction)

    # Local hourly inputs. Boundary neighbors fall back to the current hour.
    hourly["local_hourly_price_eur_mwh"] = hourly["hourly_price_eur_mwh"]
    hourly_group = hourly.groupby(DAY_KEYS)["hourly_price_eur_mwh"]
    hourly["local_prev_hour_price_eur_mwh"] = hourly_group.shift(1).fillna(hourly["hourly_price_eur_mwh"])
    hourly["local_next_hour_price_eur_mwh"] = hourly_group.shift(-1).fillna(hourly["hourly_price_eur_mwh"])
    local_columns = [c for c in hourly if c.startswith("local_")]
    features = features.merge(hourly[HOUR_KEYS + local_columns], on=HOUR_KEYS, validate="many_to_one")

    features["reference_hourly_price_eur_mwh"] = features["local_hourly_price_eur_mwh"]

    # Daily statistics and all 24 known hourly prices.
    daily = hourly.groupby(DAY_KEYS)["hourly_price_eur_mwh"].agg(
        daily_mean_price_eur_mwh="mean",
        daily_min_price_eur_mwh="min",
        daily_max_price_eur_mwh="max",
        daily_std_price_eur_mwh=lambda values: values.std(ddof=0),
    )
    daily["daily_range_price_eur_mwh"] = daily["daily_max_price_eur_mwh"] - daily["daily_min_price_eur_mwh"]
    whole_curve = hourly.pivot(index=DAY_KEYS, columns="hour_index", values="hourly_price_eur_mwh")
    whole_curve.columns = [f"daily_hour_{hour:02d}_price_eur_mwh" for hour in whole_curve.columns]
    features = features.merge(daily.join(whole_curve).reset_index(), on=DAY_KEYS, validate="many_to_one")

    # Every zone's hourly input at the same delivery date/hour.
    cross = hourly.pivot(index=["delivery_date", "hour_index"], columns="market_id", values="hourly_price_eur_mwh")
    cross.columns = [f"crosszone_{zone}_hourly_price_eur_mwh" for zone in cross.columns]
    features = features.merge(cross.reset_index(), on=["delivery_date", "hour_index"], validate="many_to_one")
    for column in cross.columns:
        zone = column.removeprefix("crosszone_").removesuffix("_hourly_price_eur_mwh")
        features[f"crosszone_{zone}_minus_local_eur_mwh"] = features[column] - features["local_hourly_price_eur_mwh"]

    # Both residual types are computed from each historical day's own references.
    history = history_15min.copy()
    history["delivery_date"] = pd.to_datetime(history["delivery_date"])
    historical_columns = [f"historical_{kind}_deviation" for kind in REFERENCE_COLUMNS]
    if history.empty:
        history = pd.DataFrame(columns=QUARTER_KEYS + historical_columns)
        history["delivery_date"] = pd.to_datetime(history["delivery_date"])
        history[historical_columns] = history[historical_columns].astype(float)
    else:
        historical_hourly = hourly_from_quarters(history)
        history = history.merge(smooth_reference(historical_hourly), on=QUARTER_KEYS, validate="one_to_one")
        history = history.merge(
            historical_hourly.rename(columns={"hourly_price_eur_mwh": REFERENCE_COLUMNS["hourly"]}),
            on=HOUR_KEYS, validate="many_to_one",
        )
        for kind, reference_column in REFERENCE_COLUMNS.items():
            history[f"historical_{kind}_deviation"] = history["price_eur_mwh"] - history[reference_column]
        history = history[QUARTER_KEYS + historical_columns]

    # Time-indexed history includes query dates; every rolling window excludes today.
    timeline = pd.concat([history[QUARTER_KEYS], features[QUARTER_KEYS]], ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history, on=QUARTER_KEYS, how="left", validate="one_to_one")
    shape_keys = ["market_id", "hour_index", "quarter_index"]
    timeline = timeline.sort_values(shape_keys + ["delivery_date"])
    for kind in REFERENCE_COLUMNS:
        historical_column = f"historical_{kind}_deviation"
        for lag in (1, 7):
            lagged = history[QUARTER_KEYS + [historical_column]].copy()
            lagged["delivery_date"] += pd.Timedelta(days=lag)
            lagged = lagged.rename(columns={historical_column: f"history_{kind}_deviation_lag_{lag}d_eur_mwh"})
            features = features.merge(lagged, on=QUARTER_KEYS, how="left", validate="one_to_one")
        for days in (7, 28):
            rolling = (
                timeline.set_index("delivery_date")
                .groupby(shape_keys)[historical_column]
                .rolling(f"{days}D", closed="left", min_periods=1)
                .agg(["mean", "median", "count"])
                .reset_index()
                .rename(columns={
                    "mean": f"history_{kind}_deviation_mean_{days}d_eur_mwh",
                    "median": f"history_{kind}_deviation_median_{days}d_eur_mwh",
                    "count": f"history_{kind}_deviation_count_{days}d",
                })
            )
            features = features.merge(rolling, on=QUARTER_KEYS, how="left", validate="one_to_one")
            count_column = f"history_{kind}_deviation_count_{days}d"
            features[count_column] = features[count_column].fillna(0).astype(int)

    # Attach alternative target labels only after all model inputs are complete.
    if target_prices is not None:
        labels = target_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
        labels["delivery_date"] = pd.to_datetime(labels["delivery_date"])
        features = features.merge(labels, on=QUARTER_KEYS, how="left", validate="one_to_one")
        for kind, reference_column in REFERENCE_COLUMNS.items():
            features[TARGET_COLUMNS[kind]] = features["price_eur_mwh"] - features[reference_column]

    feature_columns = [c for prefix in FEATURE_PREFIXES for c in features if c.startswith(prefix)]
    label_columns = [c for c in ["price_eur_mwh", *TARGET_COLUMNS.values()] if c in features]
    return features[QUARTER_KEYS + feature_columns + label_columns].sort_values(QUARTER_KEYS).reset_index(drop=True)

def get_feature_groups(feature_table):
    """One canonical vocabulary for definitions, selection and reporting."""
    groups = {}
    for group in ("position", "calendar", "local", "daily", "reference",
                  "history_hourly", "history_smooth", "crosszone"):
        groups[group] = [c for c in feature_table if c.startswith(group + "_")]
    # The hourly reference equals local_hourly_price_eur_mwh: keep one predictor.
    groups["reference"] = ["reference_smooth_price_eur_mwh"]
    columns = [c for values in groups.values() for c in values]
    if len(columns) != len(set(columns)):
        raise ValueError("Feature groups must not overlap.")
    return groups


train_hourly = hourly_from_quarters(prices)
feature_table = build_features(train_hourly, history_15min=prices, target_prices=prices)
FEATURE_GROUPS = get_feature_groups(feature_table)
display(pd.Series({group: len(columns) for group, columns in FEATURE_GROUPS.items()}, name="feature_count").to_frame())

,feature_count
position,4
calendar,8
local,3
daily,29
reference,1
history_hourly,8
history_smooth,8
crosszone,8


## 3. Helpers: output constraints and MAE

Output columns: `delivery_date, market_id, hour_index, quarter_index, predicted_price_eur_mwh`.

Checks require all supplied keys, four zones, 24 hours and 96 predictions per day/zone; exactly one finite price per quarter 1–4; no missing/extra/duplicate keys; and `abs(mean(four predictions) − hourly price) <= 0.0001 EUR/MWh`. Zero and negative prices are valid.

Preliminary price = matching reference + predicted residual. Add `hourly price − mean(four preliminary prices)` to every quarter in the hour to enforce consistency.

The official MAE averages absolute price errors **within each day/zone → across days within each zone → equally across the four zones (25% each)**. Every exported CSV is read back, checked and scored.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

EXPECTED_ZONES = ("ZONE_0", "ZONE_1", "ZONE_2", "ZONE_3")
PREDICTION_COLUMN = "predicted_price_eur_mwh"
OUTPUT_COLUMNS = QUARTER_KEYS + [PREDICTION_COLUMN]
HOURLY_TOLERANCE = 0.0001


def validate_prediction_output(predictions, hourly_inputs, tolerance=HOURLY_TOLERANCE):
    """Raise ValueError if any required schema, coverage or hourly check fails."""
    if not np.isfinite(tolerance) or not 0 <= tolerance <= HOURLY_TOLERANCE:
        raise ValueError("Tolerance cannot exceed the official 0.0001 EUR/MWh.")
    if list(predictions.columns) != OUTPUT_COLUMNS:
        raise ValueError(f"Prediction columns must be {OUTPUT_COLUMNS}, in this order.")
    pred = predictions.copy()
    hourly = hourly_inputs[HOUR_KEYS + ["hourly_price_eur_mwh"]].copy()
    for data, keys, value in (
        (pred, QUARTER_KEYS, PREDICTION_COLUMN),
        (hourly, HOUR_KEYS, "hourly_price_eur_mwh"),
    ):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"], errors="raise")
        if data.empty or data[keys].isna().any().any():
            raise ValueError("Empty input or missing identifier.")
        if data.duplicated(keys).any():
            raise ValueError("Duplicate date/zone/time keys.")
        if not data["hour_index"].isin(range(1, 25)).all():
            raise ValueError("Hour indices must be integers 1–24.")
        if (not pd.api.types.is_numeric_dtype(data[value])
                or pd.api.types.is_bool_dtype(data[value])
                or not np.isfinite(data[value]).all()):
            raise ValueError("Prices must be finite numeric values.")
    if not pred["quarter_index"].isin([1, 2, 3, 4]).all():
        raise ValueError("Quarter indices must be integers 1–4.")
    if not hourly.groupby(DAY_KEYS).size().eq(24).all():
        raise ValueError("Hourly input must have all 24 hours per day/zone.")
    zone_sets = hourly.groupby("delivery_date")["market_id"].agg(lambda values: set(values))
    if not zone_sets.map(lambda zones: zones == set(EXPECTED_ZONES)).all():
        raise ValueError("Hourly input must contain the four official zones for every date.")

    expected_keys = hourly[HOUR_KEYS].merge(pd.DataFrame({"quarter_index": [1, 2, 3, 4]}), how="cross")
    coverage = expected_keys.merge(pred[QUARTER_KEYS], on=QUARTER_KEYS, how="outer", indicator=True, validate="one_to_one")
    if not coverage["_merge"].eq("both").all():
        raise ValueError("Missing or unexpected prediction keys.")
    if not pred.groupby(HOUR_KEYS).size().eq(4).all():
        raise ValueError("Each hour needs exactly four quarters.")
    if not pred.groupby(DAY_KEYS).size().eq(96).all():
        raise ValueError("Each day/zone needs exactly 96 predictions.")
    averages = pred.groupby(HOUR_KEYS, as_index=False)[PREDICTION_COLUMN].mean()
    averages = averages.merge(hourly, on=HOUR_KEYS, validate="one_to_one")
    hourly_errors = (averages[PREDICTION_COLUMN] - averages["hourly_price_eur_mwh"]).abs()
    if not np.isfinite(hourly_errors).all():
        raise ValueError("Calculated hourly consistency errors must be finite.")
    max_error = hourly_errors.max()
    if max_error > tolerance:
        raise ValueError(f"Hourly consistency error {max_error:.10g} exceeds {tolerance} EUR/MWh.")
    return {
        "constraints_passed": True,
        "prediction_rows": len(pred),
        "prediction_days": pred["delivery_date"].nunique(),
        "max_hourly_error_eur_mwh": float(max_error),
    }


def make_predictions(feature_rows, hourly_inputs, predicted_deviations=None, *, reference_type="smooth"):
    """Reconstruct prices using the specified reference, then enforce hourly means."""
    if reference_type not in REFERENCE_COLUMNS:
        raise ValueError("reference_type must be hourly or smooth.")
    deviations = np.zeros(len(feature_rows)) if predicted_deviations is None else np.asarray(predicted_deviations, dtype=float)
    if deviations.shape != (len(feature_rows),) or not np.isfinite(deviations).all():
        raise ValueError("Need one finite predicted deviation per feature row.")
    output = feature_rows[QUARTER_KEYS].copy()
    output[PREDICTION_COLUMN] = feature_rows[REFERENCE_COLUMNS[reference_type]].to_numpy() + deviations
    hourly = hourly_inputs[HOUR_KEYS + ["hourly_price_eur_mwh"]].copy()
    hourly["delivery_date"] = pd.to_datetime(hourly["delivery_date"])
    output = output.merge(hourly, on=HOUR_KEYS, how="left", validate="many_to_one")
    preliminary_means = output.groupby(HOUR_KEYS)[PREDICTION_COLUMN].transform("mean")
    output[PREDICTION_COLUMN] += output["hourly_price_eur_mwh"] - preliminary_means
    output = output[OUTPUT_COLUMNS].sort_values(QUARTER_KEYS).reset_index(drop=True)
    validate_prediction_output(output, hourly_inputs)
    return output


def challenge_mae(predictions, actual_prices, hourly_inputs):
    """Official date -> zone -> equally weighted four-zone MAE."""
    validate_prediction_output(predictions, hourly_inputs)
    pred = predictions.copy()
    truth = actual_prices[QUARTER_KEYS + ["price_eur_mwh"]].copy()
    for data in (pred, truth):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"])
    if truth[QUARTER_KEYS].isna().any().any() or truth.duplicated(QUARTER_KEYS).any():
        raise ValueError("Observed labels have missing or duplicate keys.")
    if not pd.api.types.is_numeric_dtype(truth["price_eur_mwh"]) or not np.isfinite(truth["price_eur_mwh"]).all():
        raise ValueError("Observed labels must be finite numeric prices.")
    errors = pred.merge(truth, on=QUARTER_KEYS, how="outer", indicator=True, validate="one_to_one")
    if not errors["_merge"].eq("both").all():
        raise ValueError("Predictions and observed labels must have exactly matching keys.")
    errors["absolute_error_eur_mwh"] = (errors[PREDICTION_COLUMN] - errors["price_eur_mwh"]).abs()
    by_day_zone = errors.groupby(DAY_KEYS, as_index=False)["absolute_error_eur_mwh"].mean().rename(columns={"absolute_error_eur_mwh": "mae_eur_mwh"})
    by_zone = by_day_zone.groupby("market_id", as_index=False)["mae_eur_mwh"].mean()
    by_zone = by_zone.set_index("market_id").loc[list(EXPECTED_ZONES)].reset_index()
    return {
        "mae_eur_mwh": float(by_zone["mae_eur_mwh"].mean()),
        "by_day_zone": by_day_zone,
        "by_zone": by_zone,
    }


def evaluate_experiment(name, feature_rows, hourly_inputs, actual_prices,
                        predicted_deviations=None, *, reference_type="smooth",
                        output_dir=Path("outputs/experiments")):
    """Correct, validate, export, read back, score and display one experiment."""
    predictions = make_predictions(feature_rows, hourly_inputs, predicted_deviations, reference_type=reference_type)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"{name}_predictions.csv"
    predictions.to_csv(path, index=False, float_format="%.10f", date_format="%Y-%m-%d")
    predictions = pd.read_csv(path, parse_dates=["delivery_date"])
    checks = validate_prediction_output(predictions, hourly_inputs)
    scores = challenge_mae(predictions, actual_prices, hourly_inputs)
    scores["by_day_zone"].to_csv(output_dir / f"{name}_mae_by_day_zone.csv", index=False)
    summary = {"experiment": name, "reference_type": reference_type, "mae_eur_mwh": scores["mae_eur_mwh"], **checks, "prediction_file": str(path)}
    print(f"{name}: MAE = {scores['mae_eur_mwh']:.6f} EUR/MWh | constraints: PASS | max hourly error = {checks['max_hourly_error_eur_mwh']:.3g}")
    return {"summary": summary, "predictions": predictions, "scores": scores}

def historical_residual_profile(query_rows, historical_residuals, *, reference_type="smooth"):
    """Average past residuals per zone/hour/quarter; exclude each query date.

    Each day contributes at most one residual to a profile position.
    Missing history falls back to zero deviation from the selected reference.
    """
    if reference_type not in TARGET_COLUMNS:
        raise ValueError("reference_type must be hourly or smooth.")
    target_column = TARGET_COLUMNS[reference_type]
    shape_keys = ["market_id", "hour_index", "quarter_index"]
    queries = query_rows[QUARTER_KEYS].copy()
    history = historical_residuals[QUARTER_KEYS + [target_column]].copy()
    for data in (queries, history):
        data["delivery_date"] = pd.to_datetime(data["delivery_date"])
        if data[QUARTER_KEYS].isna().any().any() or data.duplicated(QUARTER_KEYS).any():
            raise ValueError("Profile inputs require unique, non-missing quarter-hour keys.")
    if not np.isfinite(history[target_column]).all():
        raise ValueError("Historical residuals must be finite.")
    timeline = pd.concat([history[QUARTER_KEYS], queries], ignore_index=True).drop_duplicates()
    timeline = timeline.merge(history, on=QUARTER_KEYS, how="left", validate="one_to_one")
    timeline = timeline.sort_values(shape_keys + ["delivery_date"])
    residuals = timeline.groupby(shape_keys)[target_column]
    # Shift within the same zone/hour/quarter BEFORE aggregating over dates.
    timeline["profile_residual_sum_eur_mwh"] = residuals.transform(
        lambda values: values.shift(1).expanding(min_periods=1).sum()
    ).fillna(0)
    timeline["profile_observation_count"] = residuals.transform(
        lambda values: values.shift(1).expanding(min_periods=1).count()
    ).fillna(0).astype(int)
    timeline["profile_mean_deviation_eur_mwh"] = (
        timeline["profile_residual_sum_eur_mwh"]
        / timeline["profile_observation_count"].replace(0, np.nan)
    ).fillna(0)
    profile_columns = [
        "profile_residual_sum_eur_mwh", "profile_observation_count",
        "profile_mean_deviation_eur_mwh",
    ]
    # Preserve query order to align the returned deviations with feature rows.
    return queries.merge(timeline[QUARTER_KEYS + profile_columns], on=QUARTER_KEYS,
                         how="left", validate="one_to_one")


## 4. Experiments / models - BASELINE

Train on the first 303 available dates, through August 1, 2026. Score the final 60 dates, August 2–September 30. Models stay fixed during validation; history features and profile averages can use actual observations from strictly earlier dates, including earlier validation dates.

Each experiment below is run separately for **hourly** and **smooth** residual targets. Both models use the identical predictor columns selected from `FEATURE_GROUPS`.

| Experiment | Predictor groups |
| --- | --- |
| `calendar` | `position` + `calendar` |
| `calendar_hourly` | Above + `local` + `daily` + `reference` |
| `calendar_hourly_history` | Above + the target's matching `history_hourly` or `history_smooth` group |
| `calendar_hourly_both_histories` | Above + the other history group |
| `all` | Above + `crosszone` |
| `history_only` | Only the target's matching history group, as a separate comparison |

Even when reference prices are excluded from model predictors, the matching reference is still used to reconstruct final prices. All fitting uses the official training data. The final holdout is a model-comparison set; settings here are fixed, not optimized against it.

In [5]:
import numpy as np
from scipy import sparse
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

CALENDAR_INPUTS = [
    "position_zone", "position_hour", "position_quarter", "calendar_day_of_week",
    "calendar_month_sin", "calendar_month_cos",
]

def indicator_matrix(indices, width):
    indices = np.asarray(indices, dtype=int)
    return sparse.csr_matrix((np.ones(len(indices)), (np.arange(len(indices)), indices)), shape=(len(indices), width))

def ridge_design(data, extra_columns):
    """Known calendar basis plus quarter interactions for the selected inputs."""
    quarter = data["position_quarter"].to_numpy(dtype=int) - 1
    hour_quarter = (data["position_hour"].to_numpy(dtype=int) - 1) * 4 + quarter
    weekday_quarter = data["calendar_day_of_week"].to_numpy(dtype=int) * 4 + quarter
    zone_number = data["position_zone"].map({zone: i for i, zone in enumerate(EXPECTED_ZONES)}).to_numpy(dtype=int)
    hq = indicator_matrix(hour_quarter, 96)
    q = indicator_matrix(quarter, 4)
    blocks = [
        hq, indicator_matrix(weekday_quarter, 28), indicator_matrix(zone_number * 4 + quarter, 16),
        hq.multiply(data["calendar_month_sin"].to_numpy()[:, None]),
        hq.multiply(data["calendar_month_cos"].to_numpy()[:, None]),
    ]
    missing_indicators = [c for c in data.columns if c.startswith("missingindicator_")]
    for column in list(extra_columns) + missing_indicators:
        values = data[column].to_numpy(dtype=float)
        if column.startswith("history_") and column.endswith("_eur_mwh"):
            blocks.append(sparse.csr_matrix(values[:, None]))
        else:
            blocks.append(q.multiply(values[:, None]))
    return sparse.hstack(blocks, format="csr")

def make_ridge_model(columns, alpha=10.0):
    if "position_quarter" not in columns:
        return Pipeline([
            ("imputation", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
            ("scaling", StandardScaler()),
            ("regressor", Ridge(alpha=alpha, solver="cholesky")),
        ])
    numeric = [c for c in columns if c != "position_zone"]
    imputation = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), numeric),
    ], remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
    return Pipeline([
        ("imputation", imputation),
        ("design", FunctionTransformer(ridge_design, validate=False,
             kw_args={"extra_columns": [c for c in columns if c not in CALENDAR_INPUTS]})),
        ("scaling", StandardScaler(with_mean=False)),
        ("regressor", Ridge(alpha=alpha, solver="lsqr", tol=1e-6)),
    ])


def make_catboost_model(columns):
    return CatBoostRegressor(
        iterations=500, depth=6, learning_rate=0.05,
        loss_function="MAE", eval_metric="MAE",
        cat_features=["position_zone"] if "position_zone" in columns else [],
        random_seed=42, thread_count=4, allow_writing_files=False, verbose=False,
    )


def run_models(model_name, training_rows, validation_rows, experiments_by_target,
               validation_hourly, validation_truth):
    results, fitted_models = {}, {}
    for target, configurations in experiments_by_target.items():
        for experiment, columns in configurations.items():
            name = f"{model_name}_{target}_{experiment}"
            model = make_ridge_model(columns) if model_name == "ridge" else make_catboost_model(columns)
            model.fit(training_rows[columns], training_rows[TARGET_COLUMNS[target]])
            result = evaluate_experiment(
                name, validation_rows, validation_hourly, validation_truth,
                predicted_deviations=model.predict(validation_rows[columns]), reference_type=target,
            )
            result["summary"].update(model=model_name, feature_set=experiment, feature_count=len(columns))
            results[name], fitted_models[name] = result, model
    return results, fitted_models


import json

validation_start = pd.Timestamp(np.sort(feature_table.delivery_date.unique())[-60])
fit_rows = feature_table[feature_table.delivery_date < validation_start].reset_index(drop=True)
validation_hourly = train_hourly[train_hourly.delivery_date >= validation_start].copy()
validation_truth = prices[prices.delivery_date >= validation_start].copy()
validation_rows = build_features(validation_hourly, history_15min=prices)

def select_groups(*names):
    return [column for name in names for column in FEATURE_GROUPS[name]]

EXPERIMENTS = {}
for target in REFERENCE_COLUMNS:
    own, other = f"history_{target}", f"history_{'smooth' if target == 'hourly' else 'hourly'}"
    calendar = ("position", "calendar")
    hourly = calendar + ("local", "daily", "reference")
    EXPERIMENTS[target] = {
        "calendar": select_groups(*calendar),
        "calendar_hourly": select_groups(*hourly),
        "calendar_hourly_history": select_groups(*hourly, own),
        "calendar_hourly_both_histories": select_groups(*hourly, own, other),
        "all": select_groups(*hourly, own, other, "crosszone"),
        "history_only": select_groups(own),
    }

output_dir = Path("outputs/experiments")
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / "feature_sets.json").write_text(json.dumps(EXPERIMENTS, indent=2) + "\n")
experiments, models = {}, {}
print(f"Training: {fit_rows.delivery_date.nunique()} dates; validation: {validation_rows.delivery_date.nunique()} dates")

Training: 303 dates; validation: 60 dates


### 4.1 Baselines

For each reference: (1) use the reference alone; (2) add the average residual from all earlier dates for the same zone/hour/quarter (96 averages per zone). With no history, use zero residual.

In [6]:
for target in REFERENCE_COLUMNS:
    for method in ("reference", "historical_profile"):
        deviations = None
        if method == "historical_profile":
            profile = historical_residual_profile(validation_rows, feature_table, reference_type=target)
            deviations = profile["profile_mean_deviation_eur_mwh"].to_numpy()
        name = f"baseline_{target}_{method}"
        experiments[name] = evaluate_experiment(
            name, validation_rows, validation_hourly, validation_truth,
            predicted_deviations=deviations, reference_type=target,
        )
        experiments[name]["summary"].update(model="baseline", feature_set=method, feature_count=0)

baseline_hourly_reference: MAE = 9.753915 EUR/MWh | constraints: PASS | max hourly error = 5.68e-14
baseline_hourly_historical_profile: MAE = 6.828230 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
baseline_smooth_reference: MAE = 6.495620 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
baseline_smooth_historical_profile: MAE = 5.319968 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11


### 4.2 Ridge regression

Fixed `alpha=10`. Calendar inputs are encoded as hour × quarter, weekday × quarter, zone × quarter and month sine/cosine × hour × quarter. Other calendar/hourly/context values are interacted with quarter; historical residual values enter directly. This lets effects change the within-hour shape. Imputation and scaling use training dates only. The `history_only` experiment uses the selected numeric history columns directly.

In [7]:
ridge_results, ridge_models = run_models(
    "ridge", fit_rows, validation_rows, EXPERIMENTS, validation_hourly, validation_truth,
)
experiments.update(ridge_results)
models.update(ridge_models)

ridge_hourly_calendar: MAE = 6.777083 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
ridge_hourly_calendar_hourly: MAE = 5.388596 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
ridge_hourly_calendar_hourly_history: MAE = 5.360577 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
ridge_hourly_calendar_hourly_both_histories: MAE = 5.197943 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
ridge_hourly_all: MAE = 5.213543 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
ridge_hourly_history_only: MAE = 6.216592 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
ridge_smooth_calendar: MAE = 5.307619 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
ridge_smooth_calendar_hourly: MAE = 5.342185 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
ridge_smooth_calendar_hourly_history: MAE = 5.171910 EUR/MWh | constraints: PASS | max hourly error = 2.51e-11
ridge_smooth_calendar_hourly_both_histories: MAE = 5.150703 EUR/MWh | constr

### 4.3 CatBoost

500 iterations, depth 6, learning rate 0.05, MAE loss, seed 42. Uses the same selected input columns as Ridge; zone is categorical when present. Interactions and numeric missing values are handled by CatBoost. No validation labels are passed to `fit`.

In [8]:
catboost_results, catboost_models = run_models(
    "catboost", fit_rows, validation_rows, EXPERIMENTS, validation_hourly, validation_truth,
)
experiments.update(catboost_results)
models.update(catboost_models)

catboost_hourly_calendar: MAE = 6.705284 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_hourly_calendar_hourly: MAE = 5.418144 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_hourly_calendar_hourly_history: MAE = 6.734017 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_hourly_calendar_hourly_both_histories: MAE = 6.668349 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_hourly_all: MAE = 5.956357 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_hourly_history_only: MAE = 6.131472 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_smooth_calendar: MAE = 5.207788 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_smooth_calendar_hourly: MAE = 5.201000 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_smooth_calendar_hourly_history: MAE = 5.331624 EUR/MWh | constraints: PASS | max hourly error = 2.5e-11
catboost_smooth_calendar_hourly_both_histories: MAE =

### 4.4 Summary

Validation MAE is in EUR/MWh; lower is better. All experiments use the same dates, references, hourly correction and official scoring. The audit verifies every exported prediction file.

In [9]:
results = pd.DataFrame([result["summary"] for result in experiments.values()])
results.to_csv(output_dir / "results.csv", index=False)
baselines = results[results.model.eq("baseline")].pivot(
    index="reference_type", columns="feature_set", values="mae_eur_mwh",
)
comparison = results[~results.model.eq("baseline")].pivot(
    index="feature_set", columns=["model", "reference_type"], values="mae_eur_mwh",
).reindex(list(EXPERIMENTS["smooth"]))
display(baselines.round(6))
display(comparison.round(6))

checks = []
for result in results.itertuples(index=False):
    prediction = pd.read_csv(result.prediction_file, parse_dates=["delivery_date"])
    report = validate_prediction_output(prediction, validation_hourly)
    mae = challenge_mae(prediction, validation_truth, validation_hourly)["mae_eur_mwh"]
    np.testing.assert_allclose(mae, result.mae_eur_mwh, atol=1e-10, rtol=0)
    checks.append({"experiment": result.experiment, **report, "mae_eur_mwh": mae})
audit = pd.DataFrame(checks)
audit.to_csv(output_dir / "requirements_audit.csv", index=False)
print(f"All {len(audit)} exported experiments pass. Maximum hourly error: {audit.max_hourly_error_eur_mwh.max():.3g} EUR/MWh.")
best = results.loc[results.mae_eur_mwh.idxmin()]
print(f"Best validation MAE: {best.mae_eur_mwh:.6f} — {best.experiment}")

feature_set,historical_profile,reference
reference_type,,
hourly,6.828230,9.753915
smooth,5.319968,6.495620


model                              ridge            catboost          
reference_type                    hourly    smooth    hourly    smooth
feature_set                                                           
calendar                        6.777083  5.307619  6.705284  5.207788
calendar_hourly                 5.388596  5.342185  5.418144  5.201000
calendar_hourly_history         5.360577  5.171910  6.734017  5.331624
calendar_hourly_both_histories  5.197943  5.150703  6.668349  5.537589
all                             5.213543  5.174920  5.956357  5.128286
history_only                    6.216592  5.146096  6.131472  5.068866

All 28 exported experiments pass. Maximum hourly error: 2.51e-11 EUR/MWh.
Best validation MAE: 5.068866 — catboost_smooth_history_only
